# 03 — Data quality and bank–quarter panel
BFT65B · Student ID 11230945

Read the verified Bronze source, audit financial reporting and institutional events, construct the Silver panel, and verify its lineage. No modelling labels or fitted preprocessing are produced.

## 1. Repository and pinned runtime
Use a CPU runtime. Restore the repository and pinned dependencies before running the research commands.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
REPO_URL = "https://github.com/NguyenDuong0508/11230945.git"
REPO = Path("/content/11230945")
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
else:
    assert subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=REPO, text=True).strip() == REPO_URL
    subprocess.run(["git", "pull", "--ff-only"], cwd=REPO, check=True)
print("Git revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements.txt")], check=True)
def command(script, *args):
    subprocess.run([sys.executable, str(REPO / script), *map(str, args)], cwd=REPO, check=True)


## 2. Mandatory offline validation
Run the complete repository suite, including event-policy regression tests and Bronze-to-Silver Parquet integration. No skipped tests are accepted.

In [ ]:
import xml.etree.ElementTree as ET
TESTS_PASSED = False
result_path = Path("/content/part2_tests.xml")
result_path.unlink(missing_ok=True)
result = subprocess.run([sys.executable, "-m", "pytest", "-q", "-ra", "--tb=short", "--show-capture=no", f"--junitxml={result_path}", "tests"], cwd=REPO,
                        env=dict(os.environ, PYTEST_DISABLE_PLUGIN_AUTOLOAD="1"), capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
if not result_path.is_file():
    raise RuntimeError("Missing pytest report")
cases = list(ET.parse(result_path).getroot().iter("testcase"))
problems = [(case, node) for case in cases for node in list(case) if node.tag in ("failure", "error", "skipped")]
for case, node in problems:
    print(case.get("classname"), case.get("name"), node.tag, node.get("message"), node.text or "")
if result.returncode or not cases or problems:
    raise RuntimeError("Offline validation failed; see diagnostics above")
TESTS_PASSED = True
print(f"Mandatory offline validation: PASS ({len(cases)} tests, zero skips)")


## 3. Resolve and verify the Part 1 source
The active source pointer and the actual Bronze artifacts must agree. Do not copy a run ID or retrieve a different source snapshot.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
WORKSPACE = Path("/content/drive/MyDrive/11230945")
assert TESTS_PASSED
assert not (WORKSPACE / "pending_source.json").exists(), "Complete acquisition first"
command("run_part1.py", "verify", "--workspace", WORKSPACE)
source_pointer = json.loads((WORKSPACE / "active_source.json").read_text())
SOURCE = WORKSPACE / source_pointer["path"]
print("Bronze manifest SHA-256:", source_pointer["manifest_sha256"])


## 4. Build the Silver panel and event ledger
The build processes observed quarters in chronological order. It retains all source observations, masks only invalid asset denominators and nonfinite values, and retains financial quality flags. Conditional income differences and capital sensitivity statistics are not evidence of verified reporting semantics. Institutional and failure references are stored separately.

In [ ]:
assert TESTS_PASSED
command("run_part2.py", "build", "--workspace", WORKSPACE)


## 5. Verify lineage and reconcile the panel
Recompute the deterministic transformations from Bronze, verify keys, values, schemas and reference ledgers, and check SHA-256 evidence. No downstream task uses an unverified Silver run.

In [ ]:
command("run_part2.py", "verify", "--workspace", WORKSPACE)
import hashlib
pointer = json.loads((WORKSPACE / "active_panel.json").read_text())
SILVER = WORKSPACE / pointer["path"]
with (SILVER / "manifest.json").open("rb") as f:
    assert hashlib.file_digest(f,"sha256").hexdigest() == pointer["manifest_sha256"]
manifest = json.loads((SILVER / "manifest.json").read_text())
assert manifest["state"] == "VERIFIED"
assert manifest["source_manifest_sha256"] == source_pointer["manifest_sha256"]
assert manifest["summary"]["quarters"] == 170
assert manifest["summary"]["source_rows"] == manifest["summary"]["panel_rows"]
print("Silver manifest SHA-256:", pointer["manifest_sha256"])
print(json.dumps(manifest["summary"], indent=2))


## 6. Financial data quality and treatment
Inspect missingness, zeros, flagged observations and the documented treatment policy. Flag counts overlap; do not sum them as a count of invalid rows.

In [ ]:
import pandas as pd
from IPython.display import display, Markdown, Image, HTML

def read(name):
    return pd.read_csv(SILVER / name)

def show(title, frame, formats=None):
    display(Markdown("**" + title + "**"))
    styled = frame.style.hide(axis="index").format(formats or {}, na_rep="—")
    display(styled)
    display(HTML('<div style="height:18px"></div>'))

def figure(title, name, width=680):
    display(Markdown("**" + title + "**"))
    display(Image(filename=str(SILVER / name), width=width))
    display(HTML('<div style="height:22px"></div>'))

fields = read("field_summary.csv").sort_values("missing_pct", ascending=False)
show("Table 1. Source field quality", fields, {"rows":"{:,}","missing":"{:,}","zeros":"{:,}","negative":"{:,}","nonfinite":"{:,}","missing_pct":"{:.2f}","zero_pct":"{:.2f}"})
flags = read("quality_flags.csv").groupby("rule", as_index=False)[["rows","flagged"]].sum()
flags["flagged_pct"] = 100*flags.flagged/flags.rows
show("Table 2. Automated quality flags", flags.merge(read("quality_policy.csv"),on="rule",validate="one_to_one"), {"rows":"{:,}","flagged":"{:,}","flagged_pct":"{:.3f}"})
figure("Figure 1. Observed banks by reporting quarter", "coverage.png")
figure("Figure 2. Source field missingness", "missingness.png", 580)
figure("Figure 3. Historical missingness by year", "historical_missingness.png")


## 7. Capital reporting and income-statement semantics
Zero capital ratios are retained in the primary panel. The separate sensitivity count treats zeros as missing without claiming that they represent CBLR elections. Income differences are conditional on a YTD basis and exact consecutive same-year reports. Do not treat them as verified quarterly flows across all regimes.

In [ ]:
capital = read("capital_audit.csv").groupby(["regime","field"],as_index=False)[["rows","zeros","missing","sensitivity_nonnull"]].sum()
capital["zero_pct"] = 100*capital.zeros/capital.rows
capital["missing_pct"] = 100*capital.missing/capital.rows
show("Table 3. Capital reporting and zero-as-missing sensitivity", capital, {"zero_pct":"{:.2f}","missing_pct":"{:.2f}"})
flows = read("income_semantics_audit.csv").groupby(["reporting_regime","field"],as_index=False)[["source_nonnull","candidate_nonnull","missing_predecessor","candidate_negative"]].sum()
show("Table 4. Conditional single-quarter income coverage", flows)
figure("Figure 4. Capital zero and missing rates by quarter", "capital_reporting.png")


## 8. Institutional events and exit evidence
Distinguish explicit failure and assistance records from unknown resolution types and candidate institutional ends. A retrospective audit describes full-history inconsistencies; it is not a filter on the financial panel. Event counts are not counts of distinct failed institutions.

In [ ]:
show("Table 5. Event ledger reconciliation", read("event_summary.csv"), {"records":"{:,}"})
show("Table 6. Retrospective institutional audit", read("terminal_summary.csv"), {"institutions":"{:,}"})
print(json.dumps(json.loads((SILVER/"event_diagnostics.json").read_text()),indent=2))


## 9. Point-in-time and outcome-policy diagnostics
The lag analysis describes source report availability under explicit assumptions; it does not establish historical filing timestamps or define the modelling risk set. The horizon diagnostic uses the separate event history and checks invariance to events after the outcome window.

In [ ]:
timing = read("availability_sensitivity.csv")
import numpy as np
timing["quarter_offset"] = (pd.to_datetime(timing.decision_quarter).dt.year*4+pd.to_datetime(timing.decision_quarter).dt.quarter)-(pd.to_datetime(timing.report_quarter).dt.year*4+pd.to_datetime(timing.report_quarter).dt.quarter)
timing = timing.groupby(["lag_days","quarter_offset"],as_index=False)[["available_rows","report_rows"]].sum()
timing["available_pct"] = 100*timing.available_rows/timing.report_rows
show("Table 7. Report availability under alternative lags",timing,{"available_rows":"{:,}","report_rows":"{:,}","available_pct":"{:.2f}"})
case = json.loads((SILVER/"horizon_case.json").read_text())
print("Horizon diagnostic:",json.dumps(case,indent=2))
if case["observed_report"]:
    assert case["invariant"], "Horizon invariance failed"
else:
    print("Source report absent; synthetic regression tests provide the policy check.")


## 10. Handoff
The active Silver pointer identifies the verified source and artifacts for Part 3 and Part 4. Part 3 reads the financial panel only. Part 4 establishes the decision-date risk set and final horizon-specific labels from the separate dated event evidence. Never use the retrospective terminal audit as a predictor or a blanket eligibility filter. Save the executed notebook with summary outputs; do not commit source or Silver data.